# Community-aware Influence Maximization

**UPDATED:** đổi `DATASET_NAME` trong cell Setup để chọn dataset trong `networks/`.

Notebook này chạy bài toán Influence Maximization trên graph text.

Pipeline:

1. Nạp graph từ `networks/<dataset>.txt` hoặc `networks/<dataset>.edgelist`.
2. Phát hiện community bằng Louvain, fallback sang greedy modularity.
3. Chia budget seed theo community.
4. Chọn seed ban đầu bằng PageRank.
5. Refine seed set bằng evolutionary search và đánh giá bằng Independent Cascade.


## 1. Setup

In [22]:
from pathlib import Path
from dataclasses import dataclass
from statistics import mean
import random
import numpy as np
import pandas as pd
import networkx as nx

SEED = 40
# Có thể paste 1 trong 3 dạng:
#   - tên dataset trong folder network/ (vd: 'rice_subset')
#   - đường dẫn tuyệt đối/tương đối tới file (vd: '/home/.../foo.txt')
#   - URL http(s) (vd: 'https://.../foo.edgelist')
DATASET = '/home/vhaohao/hao/dacs3/network/Email_EU_Core.txt'
try:
    DATASET_DIR = Path(__file__).parent / 'network'
except NameError:
    DATASET_DIR = Path.cwd() / 'network'
GRAPH_EXTENSIONS = ('.txt', '.edgelist')


def resolve_graph_path(dataset: str) -> Path:
    if dataset.startswith(('http://', 'https://')):
        import urllib.request
        DATASET_DIR.mkdir(exist_ok=True)
        local = DATASET_DIR / Path(dataset).name
        if not local.exists():
            urllib.request.urlretrieve(dataset, local)
        return local

    direct = Path(dataset).expanduser()
    if direct.is_file():
        return direct

    for ext in GRAPH_EXTENSIONS:
        candidate = DATASET_DIR / f'{dataset}{ext}'
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f'Không tìm thấy {dataset} (đã thử URL, path trực tiếp, và {GRAPH_EXTENSIONS} trong {DATASET_DIR})'
    )


graph_path = resolve_graph_path(DATASET)
DATASET_NAME = graph_path.stem

graph_path


PosixPath('/home/vhaohao/hao/dacs3/network/Email_EU_Core.txt')

## 2. Cấu Hình

In [23]:
@dataclass(frozen=True)
class Config:
    graph_path: Path = graph_path
    budget: int = 50
    propagation_prob: float = 0.1
    mc_runs: int = 1000
    pop_size: int = 10
    generations: int = 20
    elite_size: int = 4
    mutation_rate: float = 0.20
    seed: int = budget

config = Config()
config


Config(graph_path=PosixPath('/home/vhaohao/hao/dacs3/network/Email_EU_Core.txt'), budget=50, propagation_prob=0.1, mc_runs=1000, pop_size=10, generations=20, elite_size=4, mutation_rate=0.2, seed=50)

## 3. Load Graph Và Detect Community

In [24]:
def load_graph(file_path: Path) -> nx.Graph:
    if file_path is None:
        raise FileNotFoundError(f'Không tìm thấy graph trong {DATASET_DIR}')
    graph = nx.read_edgelist(file_path, nodetype=str)
    graph.remove_edges_from(nx.selfloop_edges(graph))
    graph.name = file_path.stem
    return graph


def graph_summary(graph: nx.Graph) -> dict:
    return {
        'name': graph.name,
        'nodes': graph.number_of_nodes(),
        'edges': graph.number_of_edges(),
        'density': nx.density(graph),
        'connected_components': nx.number_connected_components(graph),
    }


# Community detection bằng GEN-EPWOCD (thuật toán EP-WOCD tối ưu, viết bằng C++).
# Thư viện gen_epwocd nằm cùng thư mục với notebook; nó tự biên dịch epwocd.cpp
# thành binary `epwocd` (nếu chưa có / đã cũ) và gọi qua subprocess.
import sys as _sys
_MODDIR = str(DATASET_DIR.parent)
if _MODDIR not in _sys.path:
    _sys.path.insert(0, _MODDIR)
from gen_epwocd import detect_communities


graph = load_graph(config.graph_path)
communities, partition, community_method = detect_communities(graph, config.seed)

graph_summary(graph), community_method, len(communities)


({'name': 'Email_EU_Core',
  'nodes': 1005,
  'edges': 16064,
  'density': 0.031840796019900496,
  'connected_components': 20},
 'gen_epwocd',
 23)

## 4. Chọn Seed Ban Đầu

In [25]:
def weighted_choice(weights, rng: random.Random) -> int:
    total = sum(weights)
    threshold = rng.random() * total
    cumulative = 0.0
    for idx, weight in enumerate(weights):
        cumulative += weight
        if cumulative >= threshold:
            return idx
    return len(weights) - 1


def random_round_budget(communities, budget: int, rng: random.Random) -> list[int]:
    sizes = [len(c) for c in communities]
    budget = min(budget, sum(sizes))
    raw = [budget * size / sum(sizes) for size in sizes]
    allocation = [int(x) for x in raw]
    fractions = [x - int(x) for x in raw]
    remaining = budget - sum(allocation)

    while remaining > 0:
        weights = fractions if sum(fractions) > 0 else sizes
        idx = weighted_choice(weights, rng)
        if allocation[idx] < sizes[idx]:
            allocation[idx] += 1
            remaining -= 1
        fractions[idx] = 0.0
    return allocation

def pagerank_power_iteration(graph: nx.Graph, damping=0.85, max_iter=100, tol=1e-8):
    nodes = list(graph.nodes())
    if not nodes:
        return {}
    n = len(nodes)
    scores = {node: 1 / n for node in nodes}
    base = (1 - damping) / n

    for _ in range(max_iter):
        new_scores = {node: base for node in nodes}
        for source in nodes:
            neighbors = list(graph.neighbors(source))
            if not neighbors:
                share = damping * scores[source] / n
                for node in nodes:
                    new_scores[node] += share
            else:
                share = damping * scores[source] / len(neighbors)
                for target in neighbors:
                    new_scores[target] += share
        delta = sum(abs(new_scores[node] - scores[node]) for node in nodes)
        scores = new_scores
        if delta < tol:
            break
    return scores


def rank_nodes_by_pagerank(graph: nx.Graph, nodes) -> list[str]:
    subgraph = graph.subgraph(nodes).copy()
    if subgraph.number_of_edges() == 0:
        return sorted(subgraph.nodes())
    scores = pagerank_power_iteration(subgraph)
    return sorted(subgraph.nodes(), key=lambda node: scores[node], reverse=True)


def build_initial_seed_set(graph, communities, budget, seed):
    rng = random.Random(seed)
    quotas = random_round_budget(communities, min(budget, graph.number_of_nodes()), rng)
    seeds = []

    for community, quota in zip(communities, quotas):
        for node in rank_nodes_by_pagerank(graph, community)[:quota]:
            if node not in seeds:
                seeds.append(node)

    for node in rank_nodes_by_pagerank(graph, graph.nodes()):
        if len(seeds) >= min(budget, graph.number_of_nodes()):
            break
        if node not in seeds:
            seeds.append(node)
    return seeds, quotas


initial_seeds, quotas = build_initial_seed_set(graph, communities, config.budget, config.seed)
len(initial_seeds), quotas[:10], initial_seeds[:10]


(50,
 [15, 8, 7, 6, 6, 5, 3, 0, 0, 0],
 ['121', '107', '82', '106', '21', '62', '17', '282', '249', '105'])

## 5. Independent Cascade Và Evolutionary Search

In [26]:
def run_ic(graph, seeds, p=0.1, mc=100, rng=None):
    rng = rng or random.Random()
    total = 0
    seed_set = set(seeds)

    for _ in range(mc):
        active = set(seed_set)
        frontier = set(seed_set)
        while frontier:
            new_frontier = set()
            for source in frontier:
                for target in graph.neighbors(source):
                    if target not in active and rng.random() < p:
                        active.add(target)
                        new_frontier.add(target)
            frontier = new_frontier
        total += len(active)
    return total / mc


def complete_seed_set(seed_set, graph, budget, rng):
    seeds = list(dict.fromkeys(seed_set))
    candidates = [node for node in graph.nodes() if node not in seeds]
    rng.shuffle(candidates)
    while len(seeds) < budget and candidates:
        seeds.append(candidates.pop())
    return seeds[:budget]


def mutate(seed_set, graph, budget, mutation_rate, rng):
    seeds = list(seed_set)
    if seeds and rng.random() < mutation_rate:
        removed = seeds.pop(rng.randrange(len(seeds)))
        candidates = [node for node in graph.nodes() if node not in seeds and node != removed]
        if candidates:
            seeds.append(rng.choice(candidates))
    return complete_seed_set(seeds, graph, budget, rng)


def crossover(parent_a, parent_b, budget, rng):
    cut = rng.randrange(1, max(2, budget))
    child = list(parent_a[:cut])
    for node in parent_b:
        if node not in child:
            child.append(node)
        if len(child) == budget:
            break
    return child[:budget]


def evolutionary_refine(graph, initial_seeds, config):
    rng = random.Random(config.seed)
    budget = min(config.budget, graph.number_of_nodes())
    elite_size = max(1, min(config.elite_size, config.pop_size))
    population = [complete_seed_set(initial_seeds, graph, budget, rng)]
    while len(population) < config.pop_size:
        population.append(mutate(population[0], graph, budget, 1.0, rng))

    best_seed = population[0]
    best_score = run_ic(graph, best_seed, config.propagation_prob, config.mc_runs, rng)
    history = []

    for generation in range(config.generations + 1):
        scored = sorted(
            ((run_ic(graph, individual, config.propagation_prob, config.mc_runs, rng), individual)
             for individual in population),
            key=lambda item: item[0],
            reverse=True,
        )
        if scored[0][0] > best_score:
            best_score, best_seed = scored[0][0], list(scored[0][1])
        history.append({
            'generation': generation,
            'best_score': scored[0][0],
            'mean_score': mean(score for score, _ in scored),
        })

        parents = [list(individual) for _, individual in scored[:elite_size]]
        population = parents[:]
        while len(population) < config.pop_size:
            child = crossover(rng.choice(parents), rng.choice(parents), budget, rng)
            population.append(mutate(child, graph, budget, config.mutation_rate, rng))

    return best_seed, best_score, history


## 6. Chạy Pipeline

In [27]:
initial_score = run_ic(
    graph,
    initial_seeds,
    p=config.propagation_prob,
    mc=config.mc_runs,
    rng=random.Random(config.seed + 1),
)
refined_seeds, refined_score, history = evolutionary_refine(graph, initial_seeds, config)

summary = {
    'dataset': graph.name,
    'community_method': community_method,
    'num_communities': len(communities),
    'budget': len(refined_seeds),
    'initial_score': initial_score,
    'refined_score': refined_score,
    'improvement': refined_score - initial_score,
}
summary


{'dataset': 'Email_EU_Core',
 'community_method': 'gen_epwocd',
 'num_communities': 23,
 'budget': 50,
 'initial_score': 709.461,
 'refined_score': 713.063,
 'improvement': 3.6019999999999754}

## 7. Kết Quả Chi Tiết

In [28]:
community_summary = [
    {'community_id': idx, 'size': len(nodes), 'quota': quotas[idx]}
    for idx, nodes in enumerate(communities)
]

print('Graph summary:')
print(graph_summary(graph))
print('\nCommunity summary, first 20:')
print(community_summary[:20])
print('\nInitial seeds:')
print(initial_seeds)
print('\nRefined seeds:')
print(refined_seeds)
print('\nSearch history:')
print(history)


Graph summary:
{'name': 'Email_EU_Core', 'nodes': 1005, 'edges': 16064, 'density': 0.031840796019900496, 'connected_components': 20}

Community summary, first 20:
[{'community_id': 0, 'size': 308, 'quota': 15}, {'community_id': 1, 'size': 147, 'quota': 8}, {'community_id': 2, 'size': 138, 'quota': 7}, {'community_id': 3, 'size': 129, 'quota': 6}, {'community_id': 4, 'size': 108, 'quota': 6}, {'community_id': 5, 'size': 97, 'quota': 5}, {'community_id': 6, 'size': 59, 'quota': 3}, {'community_id': 7, 'size': 2, 'quota': 0}, {'community_id': 8, 'size': 2, 'quota': 0}, {'community_id': 9, 'size': 2, 'quota': 0}, {'community_id': 10, 'size': 1, 'quota': 0}, {'community_id': 11, 'size': 1, 'quota': 0}, {'community_id': 12, 'size': 1, 'quota': 0}, {'community_id': 13, 'size': 1, 'quota': 0}, {'community_id': 14, 'size': 1, 'quota': 0}, {'community_id': 15, 'size': 1, 'quota': 0}, {'community_id': 16, 'size': 1, 'quota': 0}, {'community_id': 17, 'size': 1, 'quota': 0}, {'community_id': 18, 's

## 8. Chạy Nhiều Dataset

In [29]:
def run_many(files, base_config: Config):
    rows = []
    for path in files:
        cfg = Config(
            graph_path=path,
            budget=base_config.budget,
            propagation_prob=base_config.propagation_prob,
            mc_runs=base_config.mc_runs,
            pop_size=base_config.pop_size,
            generations=base_config.generations,
            elite_size=base_config.elite_size,
            mutation_rate=base_config.mutation_rate,
            seed=base_config.seed,
        )
        g = load_graph(path)
        comms, _, method = detect_communities(g, cfg.seed)
        seeds, _ = build_initial_seed_set(g, comms, cfg.budget, cfg.seed)
        start_score = run_ic(g, seeds, cfg.propagation_prob, cfg.mc_runs, random.Random(cfg.seed + 1))
        best_seeds, best_score, _ = evolutionary_refine(g, seeds, cfg)
        rows.append({
            'dataset': path.name,
            'nodes': g.number_of_nodes(),
            'edges': g.number_of_edges(),
            'community_method': method,
            'communities': len(comms),
            'initial_score': start_score,
            'refined_score': best_score,
            'improvement': best_score - start_score,
            'seed_count': len(best_seeds),
        })
    return rows

# Bỏ comment để chạy tất cả graph trong networks/.
# all_results = run_many(network_files, config)
# all_results


In [30]:
def get_RRS(G,p):
    """
    Inputs: G:  Ex2 dataframe of directed edges. Columns: ['source','target']
            p:  Disease propagation probability
    Return: A random reverse reachable set expressed as a list of nodes
    """

    # Step 1. Select random source node
    source = random.choice(np.unique(G['source']))

    # Step 2. Get an instance of g from G by sampling edges
    g = G.copy().loc[np.random.uniform(0,1,G.shape[0]) < p]

    # Step 3. Construct reverse reachable set of the random source node
    new_nodes, RRS0 = [source], [source]
    while new_nodes:

        # Limit to edges that flow into the source node
        temp = g.loc[g['target'].isin(new_nodes)]

        # Extract the nodes flowing into the source node
        temp = temp['source'].tolist()

        # Add new set of in-neighbors to the RRS
        RRS = list(set(RRS0 + temp))

        # Find what new nodes were added
        new_nodes = list(set(RRS) - set(RRS0))

        # Reset loop variables
        RRS0 = RRS[:]

    return(RRS)

In [31]:
from collections import Counter
import time
def ris(G,k,p=0.1,mc=100):
    """
    Inputs: G:  Ex2 dataframe of directed edges. Columns: ['source','target']
            k:  Size of seed set
            p:  Disease propagation probability
            mc: Number of RRSs to generate
    Return: A seed set of nodes as an approximate solution to the IM problem
    """

    # Step 1. Generate the collection of random RRSs
    start_time = time.time()
    R = [get_RRS(G,p) for _ in range(mc)]

    # Step 2. Choose nodes that appear most often (maximum coverage greedy algorithm)
    SEED, timelapse = [], []
    for _ in range(k):

        # Find node that occurs most often in R and add to seed set
        flat_list = [item for sublist in R for item in sublist]
        seed = Counter(flat_list).most_common()[0][0]
        SEED.append(seed)

        # Remove RRSs containing last chosen seed
        R = [rrs for rrs in R if seed not in rrs]

        # Record Time
        timelapse.append(time.time() - start_time)

    return(sorted(SEED),timelapse)

In [32]:
G = pd.read_csv(graph_path, sep=' ', header=None, names=['source', 'target'])


In [33]:
def IC(G,S,p=0.1,mc=100):
    """
    Input:  G:  Ex2 dataframe of directed edges. Columns: ['source','target']
            S:  Set of seed nodes
            p:  Disease propagation probability
            mc: Number of Monte-Carlo simulations
    Output: Average number of nodes influenced by the seed nodes
    """

    # Loop over the Monte-Carlo Simulations
    spread = []
    for _ in range(mc):

        # Simulate propagation process
        new_active, A = S[:], S[:]
        while new_active:

            # Get edges that flow out of each newly active node
            temp = G.loc[G['source'].isin(new_active)]

            # Extract the out-neighbors of those nodes
            targets = temp['target'].tolist()

            # Determine those neighbors that become infected
            success  = np.random.uniform(0,1,len(targets)) < p
            new_ones = np.extract(success, targets)

            # Create a list of nodes that weren't previously activated
            new_active = list(set(new_ones) - set(A))

            # Add newly activated nodes to the set of activated nodes
            A += new_active

        spread.append(len(A))

    return(np.mean(spread))

In [34]:
ris_output = ris(G,k=50,p=0.1,mc=1000)
IC(G,ris_output[0],0.1,mc=100)

np.float64(696.5)